# Метод бакетов

Есть наблюдения

$$
u_1, \ldots, u_n,
$$

где $u_i$ — идентификатор пользователя. Один пользователь может встречаться
в данных несколько раз.

Для каждого наблюдения есть значение метрики

$$
X_1, \ldots, X_n.
$$

Например, это может быть просмотр, покупка, клик и т.д.

## Проблема

Если один пользователь создаёт несколько наблюдений, то наблюдения

$$
X_1, \ldots, X_n
$$

в общем случае нельзя считать независимыми.

Например, действия одного и того же пользователя обычно коррелируют между собой.

Дополнительная проблема — объём данных может быть очень большим.

## Идея бакетизации

Для каждого пользователя определяем бакет:

$$
b(u) = \operatorname{hash}(u) \bmod B,
$$

где $B$ — количество бакетов.

На практике часто используют сотни бакетов, например $B=200$, но универсального
оптимального значения нет. Важно, чтобы бакетов было достаточно много для
статистического анализа и чтобы пользователи распределялись между ними
достаточно равномерно.

Важно, что хэширование детерминировано: один и тот же пользователь всегда
попадает в один и тот же бакет.

Таким образом, все зависимые наблюдения одного пользователя остаются внутри
одной статистической единицы.

После этого агрегируем метрику внутри каждого бакета и получаем

$$
Y_1, \ldots, Y_B.
$$

Дальнейший статистический анализ проводится уже на уровне бакетов.
### Почему бакеты можно считать независимыми?

Если:

- разные пользователи независимы друг от друга;
- хэширование детерминировано, поэтому все наблюдения одного пользователя
  всегда попадают в один бакет;
- хэш-функция достаточно равномерно и псевдослучайно распределяет пользователей
  между бакетами, например MurmurHash3 или xxHash64,

то разные бакеты содержат непересекающиеся множества независимых пользователей.

Следовательно, агрегаты

$$
Y_1, \ldots, Y_B
$$

можно рассматривать как независимые при выполнении этих предположений.

> Важно: hash сам по себе не создаёт независимость между пользователями.
> Если между пользователями есть network effects или другие зависимости,
> они могут сохраняться и между бакетами.

## Теряется ли мощность?

Рассмотрим простой случай:

- наблюдения независимы;
- каждый пользователь встречается один раз;
- всего $n$ наблюдений;
- имеется $B$ бакетов одинакового размера;
- в каждом бакете

$$
m = \frac{n}{B}
$$

наблюдений.

Пусть

$$
Y_b =
\frac{1}{m}
\sum_{i \in b} X_i
$$

— среднее значение метрики внутри бакета.

Если

$$
\operatorname{Var}(X_i)=\sigma^2,
$$

то

$$
\operatorname{Var}(Y_b)
=
\frac{\sigma^2}{m}.
$$

Среднее по бакетам:

$$
\bar Y
=
\frac{1}{B}
\sum_{b=1}^{B}Y_b.
$$

При одинаковом размере бакетов

$$
\bar Y = \bar X.
$$

Дисперсия оценки равна

$$
\operatorname{Var}(\bar Y)
=
\frac{1}{B}
\operatorname{Var}(Y_b)
=
\frac{1}{B}
\frac{\sigma^2}{m}.
$$

Так как

$$
m=\frac{n}{B},
$$

получаем

$$
\operatorname{Var}(\bar Y)
=
\frac{1}{B}
\frac{\sigma^2}{n/B}
=
\frac{\sigma^2}{n}.
$$

Но

$$
\operatorname{Var}(\bar X)
=
\frac{\sigma^2}{n}.
$$

Следовательно,

$$
\boxed{
\operatorname{Var}(\bar Y)
=
\operatorname{Var}(\bar X)
}
$$

То есть в рассмотренном идеальном случае бакетизация сама по себе не увеличивает
дисперсию оценки и поэтому асимптотически не приводит к потере мощности.

На практике число бакетов должно быть достаточно большим. Часто используют
сотни бакетов, например $B=200$, но это не универсальная константа:
при слишком малом числе бакетов уменьшается число степеней свободы и оценка
дисперсии становится менее стабильной.